<a href="https://colab.research.google.com/github/rlaapxk/wed_2026_bigdatacomputingClass/blob/main/%EA%B8%B0%EB%A7%90%EA%B3%A0%EC%82%AC_%EA%B9%80%EC%9C%A0%ED%95%9C20250571.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Streamlit과 pyngrok 설치
!pip install streamlit pyngrok -q

In [9]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# 1. 데이터 로드 및 전처리 (조건 1)
# ==========================================
url = 'https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv'
df = pd.read_csv(url)

# [오류 방지] 원본 데이터셋 컬럼명의 앞뒤 공백 완전 제거
df.columns = df.columns.str.strip()

# 결측치 제거
df = df.dropna()

# 자율 변수 3가지 선택 및 종속변수 설정
features = ['Adult mortality', 'BMI', 'GDP']

X = df[features]
y = df['Life expectancy']

# 전체 데이터를 80% 훈련, 20% 테스트 세트로 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# [오류 방지] 인덱스 꼬임 에러를 방지하기 위해 train_test_split으로 정확히 50개만 안전하게 샘플링
X_train_sampled, _, y_train_sampled, _ = train_test_split(X_train, y_train, train_size=50, random_state=42)

# ===================================================
# 2. 파이프라인 기반 모델 3종 학습 및 저장 (조건 2)
# ===================================================
models = {
    "Linear": Pipeline([
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ]),
    "Poly": Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=3)),
        ('reg', LinearRegression())
    ]),
    "Ridge": Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=3)),
        ('reg', Ridge(alpha=1.0))
    ])
}

results_summary = {}

for name, model in models.items():
    # 모델 학습
    model.fit(X_train_sampled, y_train_sampled)

    # 예측 수행
    train_pred = model.predict(X_train_sampled)
    test_pred = model.predict(X_test)

    # 평가지표 산출
    train_r2 = r2_score(y_train_sampled, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    train_mse = mean_squared_error(y_train_sampled, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    # [오류 방지] 파이프라인에서 마지막 회귀모델 앞부분(전처리)까지만 통과시켜 생성된 총 특성(Complexity) 개수 추출
    transformed_X = model[:-1].transform(X_train_sampled[:1])
    complexity = transformed_X.shape[1]

    results_summary[name] = {
        'Train R^2': train_r2,
        'Test R^2': test_r2,
        'Train MSE': train_mse,
        'Test MSE': test_mse,
        'Complexity': complexity
    }

    print(f"\n> [{name} Model]")
    print(f" - Test R^2 : {test_r2:.4f}")
    print(f" - Test MSE : {test_mse:.4f}")
    print("-" * 50)

# =====================================================
# 3. 모델 및 기준값을 하나의 딕셔너리로 묶어 파일로 저장
# =====================================================
payload = {
    'models': models,
    'results': results_summary,
    'features': features
}
joblib.dump(payload, "life_expectancy_models.pkl")
print("✅ 3종 회귀 모델 파이프라인이 'life_expectancy_models.pkl'로 성공적으로 저장되었습니다!")


> [Linear Model]
 - Test R^2 : 0.5769
 - Test MSE : 30.0488
--------------------------------------------------

> [Poly Model]
 - Test R^2 : -12.4937
 - Test MSE : 958.3554
--------------------------------------------------

> [Ridge Model]
 - Test R^2 : 0.5986
 - Test MSE : 28.5118
--------------------------------------------------
✅ 3종 회귀 모델 파이프라인이 'life_expectancy_models.pkl'로 성공적으로 저장되었습니다!


In [4]:
%%writefile app.py
import streamlit as st
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="기대수명 예측 대시보드", layout="centered")

# 1. 저장된 pkl 데이터 로드
def load_saved_data():
    return joblib.load("life_expectancy_models.pkl")

data = load_saved_data()
models = data['models']
results = data['results']
features = data['features']

st.title("💡 파이프라인 기반 기대수명 예측 웹 서비스")
st.markdown("선형 회귀, 다항 회귀(Poly), 릿지(Ridge) 규제 모델의 예측 결과를 실시간으로 비교하고 테스트합니다.")

# ==========================================
# 2. 성능 비교 섹션 출력 (조건 3)
# ==========================================
st.write("### 📊 모델 성능 평가지표 비교")

# 성능 테이블 DataFrame 출력
results_df = pd.DataFrame(results).T
st.dataframe(results_df.style.highlight_max(axis=0, subset=['Test R^2'], color='lightgreen'))

# 성능 비교 시각화 지표 (Test R^2)
st.write("#### 📈 Test R^2 점수 비교 차트")
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results_df.index, results_df['Test R^2'], color=['#4CAF50', '#2196F3', '#FF9800'])
ax.set_ylabel("Test R^2 Score")
ax.set_title("Test R^2 Score by Regression Models")
ax.axhline(0, color='black', linewidth=0.8) # 0 기준선 추가

# 막대 위에 수치 표시
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + (0.05 if yval > 0 else -0.1), f'{yval:.2f}', ha='center', va='bottom', fontsize=10)

st.pyplot(fig)
st.markdown("---")

# ==========================================
# 3. 사이드바 설정 - 모델 선택 및 실시간 입력 (조건 4)
# ==========================================
st.sidebar.header("모델 선택 및 데이터 입력")

selected_model_name = st.sidebar.selectbox("예측에 사용할 모델을 선택하세요:", list(models.keys()))
current_pipeline = models[selected_model_name]

st.sidebar.subheader("특성(Feature) 수치 조절")
# 각 특성에 대한 슬라이더 배치 (임의의 평균적 최소/최대/기본값 설정)
input_adult_mortality = st.sidebar.slider("Adult Mortality (성인 사망률)", 1.0, 800.0, 150.0)
input_bmi = st.sidebar.slider("BMI (체질량 지수)", 1.0, 90.0, 38.0)
input_gdp = st.sidebar.slider("GDP (국내총생산)", 1.0, 120000.0, 5000.0)

# 예측할 데이터 포맷 구성
new_data = pd.DataFrame([[input_adult_mortality, input_bmi, input_gdp]], columns=features)

# ==========================================
# 4. 실시간 예측 결과 출력 (조건 4)
# ==========================================
prediction = current_pipeline.predict(new_data)[0]

st.subheader(f"선택된 모델: [{selected_model_name}]")
st.write("입력된 수치 데이터를 바탕으로 예측한 기대수명(Life Expectancy) 결과입니다.")

# 예측값을 대시보드 중앙에 큰 글씨로 강조하여 출력
st.markdown(
    f"""
    <div style='text-align: center; background-color: #f0f2f6; padding: 20px; border-radius: 10px;'>
        <h1 style='color: #FF4B4B; margin: 0;'>{prediction:.2f} 세</h1>
    </div>
    """,
    unsafe_allow_html=True
)

Writing app.py


In [11]:
from pyngrok import ngrok
import os
import time

# 1. 혹시 열려있을지 모르는 기존 터널 모두 닫기
ngrok.kill()

# 2. 인증 토큰 설정 (※ 본인의 ngrok Authtoken으로 수정하세요)
ngrok.set_auth_token("3DylbxLG8YjuOWLfG7E8ZSBRPY0_HXg8AWfvxkshqGUE2gEx")

# 3. Streamlit 실행 (서버 주소를 명시적으로 127.0.0.1로 고정하여 백그라운드 구동)
os.system("nohup streamlit run app.py --server.address 127.0.0.1 --server.port 8501 > /dev/null 2>&1 &")

# Streamlit 서버가 완전히 부팅될 때까지 3초간 대기
time.sleep(3)

# 4. ngrok 터널 연결 (8501 포트 열기)
public_url = ngrok.connect(8501, bind_tls=True)

print("=" * 60)
print(f"🎉 성공적으로 웹 서비스가 배포되었습니다! 아래 링크를 클릭하세요:\n{public_url}")
print("=" * 60)

🎉 성공적으로 웹 서비스가 배포되었습니다! 아래 링크를 클릭하세요:
NgrokTunnel: "https://gatherer-duke-ashamed.ngrok-free.dev" -> "http://localhost:8501"
